In [ ]:
!pip install -r /content/requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import nltk

from tqdm import tqdm
from google.colab import files

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

nltk.download('punkt')

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU terdeteksi: {len(gpus)} device")
    for gpu in gpus:
        print(f"  - {gpu}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Memory growth setup skipped:", e)
else:
    print("GPU TIDAK terdeteksi. Buka menu Runtime > Change runtime type > pilih T4 GPU, lalu restart & jalankan ulang dari awal.")

In [ ]:
df = pd.read_csv('dataset_final.csv')

print(f"Jumlah data awal: {len(df)}")

df.head()

In [ ]:
df.columns

In [ ]:
df = df[['full_text']]

In [ ]:
df.rename(
    columns={'full_text':'text'},
    inplace=True
)

df.head()

In [ ]:
df.dropna(inplace=True)

df.drop_duplicates(
    subset=['text'],
    inplace=True
)

print(df.shape)

In [ ]:
df['text'] = df['text'].str.lower()

In [ ]:
def cleaning(text):

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#\w+", "", text)

    text = emoji.replace_emoji(
        text,
        replace=''
    )

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

df['clean_text'] = df['text'].apply(
    cleaning
)

In [ ]:
factory = StopWordRemoverFactory()

stopword = factory.create_stop_word_remover()

df['stopword'] = df[
    'clean_text'
].apply(
    stopword.remove
)

In [ ]:
factory = StemmerFactory()

stemmer = factory.create_stemmer()

tqdm.pandas()

df['stemmed'] = df[
    'stopword'
].progress_apply(
    stemmer.stem
)

In [ ]:
positive_words = [
    # Pujian
    "bagus", "hebat", "keren", "mantap", "luar biasa",
    "terbaik", "membanggakan", "sukses", "berhasil",
    # Emosi positif
    "senang", "bahagia", "gembira", "bersyukur", "bangga",
    "semangat", "termotivasi", "terinspirasi",
    # Apresiasi
    "terimakasih", "terima kasih", "salut", "kagum",
    "recommended", "rekomen", "worth it", "memuaskan",
]
negative_words = [
    # Kata kasar
    "goblok", "tolol", "idiot", "bangsat", "kampret",
    "anjing", "bajingan", "brengsek", "bodoh", "dungu",
    # Ekspresi negatif umum
    "jelek", "buruk", "gagal", "payah", "sampah",
    "menyebalkan", "menjengkelkan", "mengecewakan",
    "benci", "marah", "kesal", "frustrasi", "stress",
    # Cyberbullying / toxic
    "bullying", "bully", "hina", "rendah", "memalukan",
]

In [ ]:
def sentiment_label(text):

    pos_score = sum(
        word in text.split()
        for word in positive_words
    )

    neg_score = sum(
        word in text.split()
        for word in negative_words
    )

    if pos_score > neg_score:
        return 'positif'

    elif neg_score > pos_score:
        return 'negatif'

    else:
        return 'netral'

In [ ]:
df['label'] = df[
    'stemmed'
].apply(
    sentiment_label
)

In [ ]:
df['label'].value_counts()

In [ ]:
df.head()

# **Plot Curves**

In [ ]:
def plot_curves(history_df, model_name, loss_ylabel='Loss', x_label='Epoch'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='x', linestyle='-.', color='blue', label='Train')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='^', linestyle='--', color='orange', label='Validation')
    axes[0].set_title(f'{model_name} - {loss_ylabel}', fontweight='bold')
    axes[0].set_xlabel(x_label)
    axes[0].set_ylabel(loss_ylabel)
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.5)

    axes[1].plot(history_df['epoch'], history_df['train_acc'], marker='o', linestyle='-', color='blue', label='Train')
    axes[1].plot(history_df['epoch'], history_df['val_acc'], marker='s', linestyle='--', color='orange', label='Validation')
    axes[1].set_title(f'{model_name} - Accuracy', fontweight='bold')
    axes[1].set_xlabel(x_label)
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# **model SVM/RF/LSTM**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import cudf

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from cuml.feature_extraction.text import TfidfVectorizer as cuTfidfVectorizer
from cuml.svm import SVC as cuSVC
from cuml.ensemble import RandomForestClassifier as cuRF

from gensim.models import Word2Vec

sns.set_style('whitegrid')

In [ ]:
df = df.dropna(subset=['stemmed']).reset_index(drop=True)

X_text = df['stemmed'].astype(str)
y = df['label']

print(f"Jumlah data setelah preprocessing: {len(df)} baris")
print(y.value_counts())

In [ ]:
class_labels = sorted(np.unique(y))
results_summary = []

def evaluate_model(model_name, y_true, y_pred, labels=class_labels):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )

    print(f"===== {model_name} =====")
    print(f"Accuracy          : {acc:.4f}")
    print(f"Precision (macro) : {precision:.4f}")
    print(f"Recall (macro)    : {recall:.4f}")
    print(f"F1-score (macro)  : {f1:.4f}\n")

    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)

    plt.title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    return {
        'model': model_name,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

## MODEL 1 — SVM (GPU, cuML) + TF-IDF — Split 80:20

In [ ]:
label_encoder_svm = LabelEncoder()
y_enc_svm = label_encoder_svm.fit_transform(y)

X_train_text_svm, X_test_text_svm, y_train_svm, y_test_svm = train_test_split(
    X_text, y_enc_svm,
    test_size=0.2,
    stratify=y_enc_svm,
    random_state=42
)

X_tr_text_svm, X_val_text_svm, y_tr_svm, y_val_svm = train_test_split(
    X_train_text_svm, y_train_svm,
    test_size=0.1,
    stratify=y_train_svm,
    random_state=42
)

tfidf_svm = cuTfidfVectorizer(max_features=10000)
X_tr_svm = tfidf_svm.fit_transform(cudf.Series(X_tr_text_svm))
X_val_svm = tfidf_svm.transform(cudf.Series(X_val_text_svm))
X_test_svm = tfidf_svm.transform(cudf.Series(X_test_text_svm))

print(f"Train: {X_tr_svm.shape[0]} | Val: {X_val_svm.shape[0]} | Test: {X_test_svm.shape[0]}")

In [ ]:
svm_gpu = cuSVC(kernel='linear', probability=False)
svm_gpu.fit(X_tr_svm, cudf.Series(y_tr_svm))

pred_tr_svm = svm_gpu.predict(X_tr_svm).get()
pred_val_svm = svm_gpu.predict(X_val_svm).get()

train_acc_svm = accuracy_score(y_tr_svm, pred_tr_svm)
val_acc_svm = accuracy_score(y_val_svm, pred_val_svm)

history_svm_df = pd.DataFrame({
    'epoch': [1],
    'train_loss': [1 - train_acc_svm],
    'val_loss': [1 - val_acc_svm],
    'train_acc': [train_acc_svm],
    'val_acc': [val_acc_svm]
})

history_svm_df

In [ ]:
import cupy as cp

pred_svm = svm_gpu.predict(X_test_svm)

if isinstance(pred_svm, cp.ndarray):
    pred_svm = pred_svm.get()

if isinstance(y_test_svm, cp.ndarray):
    y_test_svm = y_test_svm.get()

pred_svm_label = label_encoder_svm.inverse_transform(pred_svm)
y_test_svm_label = label_encoder_svm.inverse_transform(y_test_svm)

metrics_svm = evaluate_model(
    "SVM (GPU) + TF-IDF (80:20)",
    y_test_svm_label,
    pred_svm_label
)

In [ ]:
joblib.dump(svm_gpu, 'model_svm_tfidf.joblib')
joblib.dump(tfidf_svm, 'tfidf_vectorizer_svm.joblib')
joblib.dump(label_encoder_svm, 'label_encoder_svm.joblib')
print('Model SVM (GPU) + TF-IDF berhasil disimpan')

### Hyperparameter Tuning — SVM (GPU, cuML)

In [ ]:
import cupy as cp

best_svm_acc = 0
best_svm_params = None
best_svm_model = None

param_grid_svm = [
    {'C': c, 'kernel': 'linear'} for c in [0.1, 1.0, 5.0, 10.0]
]

for params in param_grid_svm:
    model = cuSVC(**params)
    model.fit(X_tr_svm, cudf.Series(y_tr_svm))

    val_pred = model.predict(X_val_svm)
    if isinstance(val_pred, cp.ndarray):
        val_pred = val_pred.get()

    acc = accuracy_score(y_val_svm, val_pred)
    print(f"Params: {params} -> Val Accuracy: {acc:.4f}")

    if acc > best_svm_acc:
        best_svm_acc = acc
        best_svm_params = params
        best_svm_model = model

print("Best params:", best_svm_params)
print("Best Val accuracy:", best_svm_acc)

svm_best = best_svm_model

pred_svm_best = svm_best.predict(X_test_svm)
if isinstance(pred_svm_best, cp.ndarray):
    pred_svm_best = pred_svm_best.get()

# Pastikan y_test_svm juga NumPy
if isinstance(y_test_svm, cp.ndarray):
    y_test_svm = y_test_svm.get()

pred_svm_best_label = label_encoder_svm.inverse_transform(pred_svm_best)
y_test_svm_label = label_encoder_svm.inverse_transform(y_test_svm)

metrics_svm_best = evaluate_model(
    'SVM (GPU) + TF-IDF (Tuned)',
    y_test_svm_label,
    pred_svm_best_label
)

results_summary.append(metrics_svm_best)

joblib.dump(svm_best, 'model_svm_tfidf_tuned.joblib')
print('Model SVM (GPU, tuned) berhasil disimpan')

## MODEL 2 — Random Forest (GPU, cuML) + Word2Vec — Split 80:20

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

# =========================
# TOKENISASI
# =========================
tokenized_text = X_text.astype(str).str.split()

# =========================
# LABEL ENCODING
# =========================
label_encoder_rf_w2v = LabelEncoder()
y_enc_rf_w2v = label_encoder_rf_w2v.fit_transform(y)

# =========================
# TRAIN-TEST SPLIT (STRATIFIED OK)
# =========================
train_idx, test_idx, y_train_rf_w2v, y_test_rf_w2v = train_test_split(
    tokenized_text.index,
    y_enc_rf_w2v,
    test_size=0.2,
    stratify=y_enc_rf_w2v,
    random_state=42
)

tokenized_train = tokenized_text.loc[train_idx]
tokenized_test = tokenized_text.loc[test_idx]

train_sentences = tokenized_train.tolist()
test_sentences = tokenized_test.tolist()

# =========================
# CLASS BALANCING (OVERSAMPLING) - HANYA DATA TRAIN
# =========================

from collections import Counter

rng = np.random.RandomState(42)

def oversample_indices(labels, rng):
    labels = np.array(labels)
    max_count = Counter(labels).most_common(1)[0][1]
    balanced_idx = []
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        if len(cls_idx) < max_count:
            extra = rng.choice(cls_idx, size=max_count - len(cls_idx), replace=True)
            cls_idx = np.concatenate([cls_idx, extra])
        balanced_idx.extend(cls_idx.tolist())
    balanced_idx = np.array(balanced_idx)
    rng.shuffle(balanced_idx)
    return balanced_idx

os_idx_w2v = oversample_indices(y_train_rf_w2v, rng)
train_sentences = [train_sentences[i] for i in os_idx_w2v]
y_train_rf_w2v = y_train_rf_w2v[os_idx_w2v]

print("Distribusi kelas train SEBELUM oversampling:", Counter(np.array(y_train_rf_w2v)[np.argsort(os_idx_w2v)]))
print("Distribusi kelas train SETELAH oversampling :", Counter(y_train_rf_w2v))

# =========================
# WORD2VEC (IMPROVED VERSION)
# =========================
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=300,
    window=10,
    min_count=1,
    sg=1,
    epochs=40,
    negative=15,
    sample=1e-4,
    workers=os.cpu_count(),
    seed=42
)

print(f"Ukuran vocabulary Word2Vec: {len(w2v_model.wv.key_to_index)}")

# =========================
# TF-IDF WEIGHTING (IMPROVED)
# =========================
tfidf_weight_vectorizer = TfidfVectorizer(
    analyzer=lambda x: x,
    min_df=1,
    max_df=0.85,
    sublinear_tf=True
)

tfidf_weight_vectorizer.fit(train_sentences)

tfidf_vocab = tfidf_weight_vectorizer.vocabulary_
tfidf_idf = tfidf_weight_vectorizer.idf_

# =========================
# DOCUMENT VECTOR (STABIL VERSION)
# =========================
def document_vector(tokens, model):

    vectors = []
    weights = []

    for word in tokens:

        if word in model.wv.key_to_index and word in tfidf_vocab:

            vectors.append(model.wv[word])
            weights.append(tfidf_idf[tfidf_vocab[word]])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    vectors = np.array(vectors)
    weights = np.array(weights)

    # 🔥 stabilisasi weights
    weights = weights / (np.sum(weights) + 1e-8)

    return np.average(vectors, axis=0, weights=weights)

# =========================
# EMBEDDING GENERATION
# =========================
X_train_w2v_vec = np.array([
    document_vector(doc, w2v_model)
    for doc in train_sentences
])

X_test_w2v_vec = np.array([
    document_vector(doc, w2v_model)
    for doc in test_sentences
])

# =========================
# SCALING (ROBUST + STABLE)
# =========================
scaler_w2v = RobustScaler()

X_train_rf_w2v = scaler_w2v.fit_transform(X_train_w2v_vec)
X_test_rf_w2v = scaler_w2v.transform(X_test_w2v_vec)

# =========================
# FINAL CHECK
# =========================
print("Shape Train :", X_train_rf_w2v.shape)
print("Shape Test  :", X_test_rf_w2v.shape)

# =========================
# CLASS DISTRIBUTION CHECK
# =========================
unique, counts = np.unique(y_train_rf_w2v, return_counts=True)
print("\nDistribusi kelas train:")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c} samples")

In [ ]:
X_tr_rf_w2v, X_val_rf_w2v, y_tr_rf_w2v, y_val_rf_w2v = train_test_split(
    X_train_rf_w2v, y_train_rf_w2v,
    test_size=0.1,
    stratify=y_train_rf_w2v,
    random_state=42
)

X_tr_rf_w2v_gpu = cudf.DataFrame(X_tr_rf_w2v.astype(np.float32))
X_val_rf_w2v_gpu = cudf.DataFrame(X_val_rf_w2v.astype(np.float32))
X_test_rf_w2v_gpu = cudf.DataFrame(X_test_rf_w2v.astype(np.float32))

tree_steps = [20, 60, 100, 160, 220, 280, 320]

history_rf_w2v = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for n_trees in tree_steps:
    rf_w2v = cuRF(n_estimators=n_trees, random_state=42)
    rf_w2v.fit(X_tr_rf_w2v_gpu, cudf.Series(y_tr_rf_w2v))

    train_pred = rf_w2v.predict(X_tr_rf_w2v_gpu).to_numpy()
    val_pred = rf_w2v.predict(X_val_rf_w2v_gpu).to_numpy()

    train_acc = accuracy_score(y_tr_rf_w2v, train_pred)
    val_acc = accuracy_score(y_val_rf_w2v, val_pred)

    history_rf_w2v['epoch'].append(n_trees)
    history_rf_w2v['train_loss'].append(1 - train_acc)
    history_rf_w2v['val_loss'].append(1 - val_acc)
    history_rf_w2v['train_acc'].append(train_acc)
    history_rf_w2v['val_acc'].append(val_acc)

history_rf_w2v_df = pd.DataFrame(history_rf_w2v)
history_rf_w2v_df

In [ ]:
plot_curves(
    history_rf_w2v_df, 'Random Forest (GPU, Word2Vec)',
    loss_ylabel='Error Rate (1 - Accuracy)',
    x_label='Jumlah Pohon (n_estimators)'
)

In [ ]:
rf_w2v_final = cuRF(n_estimators=320, random_state=42)
rf_w2v_final.fit(X_tr_rf_w2v_gpu, cudf.Series(y_tr_rf_w2v))

pred_rf_w2v = rf_w2v_final.predict(X_test_rf_w2v_gpu).to_numpy()
pred_rf_w2v_label = label_encoder_rf_w2v.inverse_transform(pred_rf_w2v)
y_test_rf_w2v_label = label_encoder_rf_w2v.inverse_transform(y_test_rf_w2v)

metrics_rf_w2v = evaluate_model('Random Forest (GPU) + Word2Vec (80:20)', y_test_rf_w2v_label, pred_rf_w2v_label)
results_summary.append(metrics_rf_w2v)

joblib.dump(rf_w2v_final, 'model_rf_word2vec.joblib')
w2v_model.save('word2vec_model.model')
joblib.dump(label_encoder_rf_w2v, 'label_encoder_rf_w2v.joblib')
print('Model Random Forest (GPU) + Word2Vec berhasil disimpan')

### Hyperparameter Tuning — Random Forest (GPU, Word2Vec)

In [ ]:
import itertools

param_grid_rf_w2v = {
    'n_estimators': [150, 250, 350],
    'max_depth': [16, 24, 32],
    'max_features': ['sqrt', 'log2']
}

best_acc_rf_w2v = 0
best_params_rf_w2v = None
best_model_rf_w2v = None

keys, values = zip(*param_grid_rf_w2v.items())
for combo in itertools.product(*values):
    params = dict(zip(keys, combo))
    model = cuRF(random_state=42, **params)
    model.fit(X_tr_rf_w2v_gpu, cudf.Series(y_tr_rf_w2v))
    val_pred = model.predict(X_val_rf_w2v_gpu).to_numpy()
    acc = accuracy_score(y_val_rf_w2v, val_pred)
    print(f"Params: {params} -> Val Accuracy: {acc:.4f}")
    if acc > best_acc_rf_w2v:
        best_acc_rf_w2v = acc
        best_params_rf_w2v = params
        best_model_rf_w2v = model

print("Best params:", best_params_rf_w2v)
print("Best Val accuracy:", best_acc_rf_w2v)

rf_w2v_best = best_model_rf_w2v
pred_rf_w2v_best = rf_w2v_best.predict(X_test_rf_w2v_gpu).to_numpy()
pred_rf_w2v_best_label = label_encoder_rf_w2v.inverse_transform(pred_rf_w2v_best)

metrics_rf_w2v_best = evaluate_model('Random Forest (GPU) + Word2Vec (Tuned)', y_test_rf_w2v_label, pred_rf_w2v_best_label)
results_summary.append(metrics_rf_w2v_best)

joblib.dump(rf_w2v_best, 'model_rf_word2vec_tuned.joblib')
print('Model RF (GPU, tuned) + Word2Vec berhasil disimpan')

## MODEL 3 — Random Forest (GPU, cuML) + TF-IDF — Split 70:30

In [ ]:
label_encoder_rf_tfidf = LabelEncoder()
y_enc_rf_tfidf = label_encoder_rf_tfidf.fit_transform(y)

X_train_text_rf, X_test_text_rf, y_train_rf_tfidf, y_test_rf_tfidf = train_test_split(
    X_text, y_enc_rf_tfidf,
    test_size=0.3,
    stratify=y_enc_rf_tfidf,
    random_state=42
)

X_tr_text_rf, X_val_text_rf, y_tr_rf_tfidf, y_val_rf_tfidf = train_test_split(
    X_train_text_rf, y_train_rf_tfidf,
    test_size=0.1,
    stratify=y_train_rf_tfidf,
    random_state=42
)

# =========================
# CLASS BALANCING (OVERSAMPLING) - HANYA DATA TRAIN
# =========================
from collections import Counter

rng_tfidf = np.random.RandomState(42)

def oversample_indices_tfidf(labels, rng):
    labels = np.array(labels)
    max_count = Counter(labels).most_common(1)[0][1]
    balanced_idx = []
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        if len(cls_idx) < max_count:
            extra = rng.choice(cls_idx, size=max_count - len(cls_idx), replace=True)
            cls_idx = np.concatenate([cls_idx, extra])
        balanced_idx.extend(cls_idx.tolist())
    balanced_idx = np.array(balanced_idx)
    rng.shuffle(balanced_idx)
    return balanced_idx

X_tr_text_rf_arr = np.array(X_tr_text_rf)
y_tr_rf_tfidf_arr = np.array(y_tr_rf_tfidf)

os_idx_tfidf = oversample_indices_tfidf(y_tr_rf_tfidf_arr, rng_tfidf)
X_tr_text_rf = X_tr_text_rf_arr[os_idx_tfidf]
y_tr_rf_tfidf = y_tr_rf_tfidf_arr[os_idx_tfidf]

print("Distribusi kelas train SETELAH oversampling:", Counter(y_tr_rf_tfidf))

tfidf_rf = cuTfidfVectorizer(max_features=15000)
X_tr_rf_tfidf = tfidf_rf.fit_transform(cudf.Series(X_tr_text_rf))
X_val_rf_tfidf = tfidf_rf.transform(cudf.Series(X_val_text_rf))
X_test_rf_tfidf = tfidf_rf.transform(cudf.Series(X_test_text_rf))

print(f"Train: {X_tr_rf_tfidf.shape[0]} | Val: {X_val_rf_tfidf.shape[0]} | Test: {X_test_rf_tfidf.shape[0]}")

In [ ]:
import numpy as np
import cudf
import pandas as pd
from sklearn.metrics import accuracy_score

# =========================
# TF-IDF sparse -> dense
# =========================
X_tr_dense = X_tr_rf_tfidf.toarray().astype(np.float32)  # cuML RF butuh float32
X_val_dense = X_val_rf_tfidf.toarray().astype(np.float32)

X_tr_gpu = cudf.DataFrame(X_tr_dense)
X_val_gpu = cudf.DataFrame(X_val_dense)

y_tr_gpu = cudf.Series(y_tr_rf_tfidf)

# =========================
# FIXED converter
# =========================
def to_numpy(x):
    import cupy as cp
    import cudf

    if isinstance(x, cp.ndarray):
        return x.get()

    if isinstance(x, (cudf.Series, cudf.DataFrame)):
        return x.to_numpy()

    return np.asarray(x)

# =========================
# TRAIN LOOP
# =========================
tree_steps = [20, 60, 100, 160, 220, 280, 320]

history_rf_tfidf = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': []
}

for n_trees in tree_steps:

    rf_tfidf = cuRF(
        n_estimators=n_trees,
        random_state=42
    )

    rf_tfidf.fit(X_tr_gpu, y_tr_gpu)

    # =========================
    # prediction (FIXED)
    # =========================
    train_pred = to_numpy(rf_tfidf.predict(X_tr_gpu))
    val_pred = to_numpy(rf_tfidf.predict(X_val_gpu))

    # =========================
    # metrics
    # =========================
    train_acc = accuracy_score(y_tr_rf_tfidf, train_pred)
    val_acc = accuracy_score(y_val_rf_tfidf, val_pred)

    history_rf_tfidf['epoch'].append(n_trees)
    history_rf_tfidf['train_loss'].append(1 - train_acc)
    history_rf_tfidf['val_loss'].append(1 - val_acc)
    history_rf_tfidf['train_acc'].append(train_acc)
    history_rf_tfidf['val_acc'].append(val_acc)

history_rf_tfidf_df = pd.DataFrame(history_rf_tfidf)
history_rf_tfidf_df

In [ ]:
plot_curves(
    history_rf_tfidf_df, 'Random Forest (GPU, TF-IDF)',
    loss_ylabel='Error Rate (1 - Accuracy)',
    x_label='Jumlah Pohon (n_estimators)'
)

In [ ]:
import cupy as cp
import cudf
import numpy as np
import joblib

# =========================
# SAFE converter
# =========================
def to_numpy(x):
    if isinstance(x, cp.ndarray):
        return x.get()
    try:
        import cudf
        if isinstance(x, (cudf.Series, cudf.DataFrame)):
            return x.to_numpy()
    except:
        pass
    return np.asarray(x)

# =========================
# 1. FIX: sparse -> dense (WAJIB)
# =========================
X_tr_dense = X_tr_rf_tfidf.toarray().astype(np.float32)  # cuML RF butuh float32
X_test_dense = X_test_rf_tfidf.toarray().astype(np.float32)

# =========================
# 2. move to GPU (cuDF)
# =========================
X_tr_gpu = cudf.DataFrame(X_tr_dense)
X_test_gpu = cudf.DataFrame(X_test_dense)

y_tr_gpu = cudf.Series(y_tr_rf_tfidf)

# =========================
# 3. Train RF
# =========================
rf_tfidf_final = cuRF(n_estimators=320, random_state=42)

rf_tfidf_final.fit(X_tr_gpu, y_tr_gpu)

# =========================
# 4. Predict (SAFE)
# =========================
pred_rf_tfidf = to_numpy(
    rf_tfidf_final.predict(X_test_gpu)
)

# =========================
# 5. Label decoding (CPU only)
# =========================
y_test_np = np.array(y_test_rf_tfidf)

pred_rf_tfidf_label = label_encoder_rf_tfidf.inverse_transform(pred_rf_tfidf)
y_test_rf_tfidf_label = label_encoder_rf_tfidf.inverse_transform(y_test_np)

# =========================
# 6. Evaluate
# =========================
metrics_rf_tfidf = evaluate_model(
    'Random Forest (GPU) + TF-IDF (70:30)',
    y_test_rf_tfidf_label,
    pred_rf_tfidf_label
)

results_summary.append(metrics_rf_tfidf)

# =========================
# 7. Save model
# =========================
joblib.dump(rf_tfidf_final, 'model_rf_tfidf.joblib')
joblib.dump(tfidf_rf, 'tfidf_vectorizer_rf.joblib')
joblib.dump(label_encoder_rf_tfidf, 'label_encoder_rf_tfidf.joblib')

print('Model Random Forest (GPU) + TF-IDF berhasil disimpan')

### Hyperparameter Tuning — Random Forest (GPU, TF-IDF, Split 70:30)

In [ ]:
import itertools
import numpy as np
import cupy as cp
import cudf
from sklearn.metrics import accuracy_score

# =========================
# SAFE converter
# =========================
def to_numpy(x):
    if isinstance(x, cp.ndarray):
        return x.get()
    try:
        import cudf
        if isinstance(x, (cudf.Series, cudf.DataFrame)):
            return x.to_numpy()
    except:
        pass
    return np.asarray(x)

# =========================
# FIX sparse -> dense (WAJIB)
# =========================
X_tr_dense = X_tr_rf_tfidf.toarray().astype(np.float32)
X_val_dense = X_val_rf_tfidf.toarray().astype(np.float32)
X_test_dense = X_test_rf_tfidf.toarray().astype(np.float32)

# =========================
# move to GPU
# =========================
X_tr_gpu = cudf.DataFrame(X_tr_dense)
X_val_gpu = cudf.DataFrame(X_val_dense)
X_test_gpu = cudf.DataFrame(X_test_dense)

y_tr_gpu = cudf.Series(y_tr_rf_tfidf)

# =========================
# GRID SEARCH
# =========================
param_grid_rf_tfidf = {
    'n_estimators': [150, 250, 350],
    'max_depth': [16, 24, 32],
    'max_features': ['sqrt', 'log2']
}

best_acc_rf_tfidf = 0
best_params_rf_tfidf = None
best_model_rf_tfidf = None

keys, values = zip(*param_grid_rf_tfidf.items())

for combo in itertools.product(*values):

    params = dict(zip(keys, combo))

    model = cuRF(random_state=42, **params)

    model.fit(X_tr_gpu, y_tr_gpu)

    # =========================
    # prediction SAFE
    # =========================
    val_pred = to_numpy(model.predict(X_val_gpu))

    # =========================
    # evaluation SAFE
    # =========================
    acc = accuracy_score(y_val_rf_tfidf, val_pred)

    print(f"Params: {params} -> Val Accuracy: {acc:.4f}")

    if acc > best_acc_rf_tfidf:
        best_acc_rf_tfidf = acc
        best_params_rf_tfidf = params
        best_model_rf_tfidf = model

print("\nBest params:", best_params_rf_tfidf)
print("Best Val accuracy:", best_acc_rf_tfidf)

# =========================
# FINAL MODEL
# =========================
rf_tfidf_best = best_model_rf_tfidf

pred_rf_tfidf_best = to_numpy(
    rf_tfidf_best.predict(X_test_gpu)
)

# =========================
# LABEL CONVERSION SAFE
# =========================
y_test_np = np.array(y_test_rf_tfidf)

pred_rf_tfidf_best_label = label_encoder_rf_tfidf.inverse_transform(pred_rf_tfidf_best)
y_test_rf_tfidf_label = label_encoder_rf_tfidf.inverse_transform(y_test_np)

# =========================
# evaluation
# =========================
metrics_rf_tfidf_best = evaluate_model(
    'Random Forest (GPU) + TF-IDF (Tuned)',
    y_test_rf_tfidf_label,
    pred_rf_tfidf_best_label
)

results_summary.append(metrics_rf_tfidf_best)

# =========================
# save
# =========================
import joblib

joblib.dump(rf_tfidf_best, 'model_rf_tfidf_tuned.joblib')
print('Model RF (GPU, tuned) + TF-IDF berhasil disimpan')

## MODEL 4 — Deep Learning (LSTM, GPU) + Split 80:20

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
X_train_text_lstm, X_test_text_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_text, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

label_encoder_lstm = LabelEncoder()
label_encoder_lstm.fit(y_train_lstm)

y_train_lstm_enc = to_categorical(label_encoder_lstm.transform(y_train_lstm))
y_test_lstm_enc = to_categorical(label_encoder_lstm.transform(y_test_lstm))

max_words = 10000
max_len = 100

tokenizer_lstm = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer_lstm.fit_on_texts(X_train_text_lstm)

X_train_seq = tokenizer_lstm.texts_to_sequences(X_train_text_lstm)
X_test_seq = tokenizer_lstm.texts_to_sequences(X_test_text_lstm)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

X_tr_lstm, X_val_lstm, y_tr_lstm, y_val_lstm = train_test_split(
    X_train_pad, y_train_lstm_enc,
    test_size=0.1,
    stratify=y_train_lstm,
    random_state=42
)

print(f"Train: {X_tr_lstm.shape[0]} | Val: {X_val_lstm.shape[0]} | Test: {X_test_pad.shape[0]}")

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(32)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

lstm_model.summary()

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history_lstm = lstm_model.fit(
    X_tr_lstm, y_tr_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=40,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
history_lstm_df = pd.DataFrame({
    'epoch': range(1, len(history_lstm.history['loss']) + 1),
    'train_loss': history_lstm.history['loss'],
    'val_loss': history_lstm.history['val_loss'],
    'train_acc': history_lstm.history['accuracy'],
    'val_acc': history_lstm.history['val_accuracy']
})
history_lstm_df

In [ ]:
plot_curves(history_lstm_df, 'LSTM Bidirectional (GPU)', loss_ylabel='Loss')

In [ ]:
pred_proba_lstm = lstm_model.predict(X_test_pad)
pred_lstm_enc = np.argmax(pred_proba_lstm, axis=1)
pred_lstm = label_encoder_lstm.inverse_transform(pred_lstm_enc)

metrics_lstm = evaluate_model('LSTM Bidirectional (GPU, 80:20)', y_test_lstm, pred_lstm)
results_summary.append(metrics_lstm)

lstm_model.save('model_lstm_tuned.keras')
joblib.dump(tokenizer_lstm, 'tokenizer_lstm.joblib')
joblib.dump(label_encoder_lstm, 'label_encoder_lstm.joblib')
print('Model LSTM (GPU) berhasil disimpan')

## Perbandingan Keempat Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# 1. Convert results
# =========================
results_df = pd.DataFrame(results_summary)

# =========================
# 2. konsistensi nama kolom
# =========================
results_df = results_df.rename(columns={
    'model': 'Model',
    'accuracy': 'Accuracy',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1-Score'
})

# =========================
# 3. ambil best model per nama
# =========================
results_df = results_df.loc[
    results_df.groupby('Model')['Accuracy'].idxmax()
].reset_index(drop=True)

# =========================
# 4. reorder kolom
# =========================
results_df = results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]

# =========================
# 5. sort
# =========================
results_df = results_df.sort_values(
    by='Accuracy',
    ascending=False
).reset_index(drop=True)

display(results_df)

# =========================
# 6. TABLE VISUALIZATION
# =========================
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

table = ax.table(
    cellText=np.round(results_df.iloc[:, 1:].values, 4),
    rowLabels=results_df['Model'].values,
    colLabels=results_df.columns[1:],
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.8)

# header styling
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#4C72B0')
        cell.set_text_props(color='white', fontweight='bold')

plt.title(
    'Ringkasan Perbandingan Model',
    fontsize=14,
    fontweight='bold',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
bars = plt.bar(results_df['Model'], results_df['Accuracy'], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.4f}', ha='center', va='bottom')

plt.title('Perbandingan Accuracy Antar Model', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## Inference — Uji dengan Kalimat Baru

In [ ]:
import numpy as np
import cupy as cp
import cudf
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================
# SAFE converter (WAJIB)
# =========================
def to_numpy(x):
    if isinstance(x, cp.ndarray):
        return x.get()
    if isinstance(x, (cudf.Series, cudf.DataFrame)):
        return x.to_numpy()
    return np.asarray(x)

# =========================
# SAMPLE DATA
# =========================
sample = [
    "pelayanannya sangat bagus",
    "hari ini saya pergi ke kampus",
    "dasar goblok sekali"
]

# =========================
# 1. SVM (GPU) + TF-IDF
# =========================
sample_gpu = cudf.Series(sample)

sample_tfidf_svm = tfidf_svm.transform(sample_gpu)

pred_sample_svm = to_numpy(
    svm_gpu.predict(sample_tfidf_svm)
)

pred_sample_svm = label_encoder_svm.inverse_transform(pred_sample_svm)

# =========================
# 2. Word2Vec + RF
# =========================
sample_tokenized = [s.split() for s in sample]

sample_w2v = np.array([
    document_vector(t, w2v_model) for t in sample_tokenized
])

sample_w2v_scaled = scaler_w2v.transform(sample_w2v)

sample_w2v_gpu = cudf.DataFrame(sample_w2v_scaled.astype(np.float32))

pred_sample_rf_w2v = to_numpy(
    rf_w2v_final.predict(sample_w2v_gpu)
)

pred_sample_rf_w2v = label_encoder_rf_w2v.inverse_transform(pred_sample_rf_w2v)

# =========================
# 3. TF-IDF + RF (GPU)
# =========================

sample_tfidf_rf = tfidf_rf.transform(cudf.Series(sample))

# sparse -> dense (WAJIB)
sample_tfidf_rf_dense = sample_tfidf_rf.toarray()

sample_tfidf_rf_gpu = cudf.DataFrame(sample_tfidf_rf_dense)

pred_sample_rf_tfidf = to_numpy(
    rf_tfidf_final.predict(sample_tfidf_rf_gpu)
)

pred_sample_rf_tfidf = label_encoder_rf_tfidf.inverse_transform(pred_sample_rf_tfidf)
# =========================
# 4. LSTM / BiLSTM
# =========================
sample_seq_lstm = tokenizer_lstm.texts_to_sequences(sample)

sample_pad_lstm = pad_sequences(
    sample_seq_lstm,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

pred_sample_lstm_proba = lstm_model.predict(sample_pad_lstm)

pred_sample_lstm = label_encoder_lstm.inverse_transform(
    np.argmax(pred_sample_lstm_proba, axis=1)
)

# =========================
# 5. OUTPUT
# =========================
for i, text in enumerate(sample):
    print(f"Teks: {text}")
    print(f"  SVM (GPU) + TF-IDF   -> {pred_sample_svm[i]}")
    print(f"  RF + Word2Vec        -> {pred_sample_rf_w2v[i]}")
    print(f"  RF + TF-IDF          -> {pred_sample_rf_tfidf[i]}")
    print(f"  LSTM BiLSTM          -> {pred_sample_lstm[i]}")
    print()